# 04 · Data Analysis with Python
**Target:** `freight_value` · 
**Đơn vị phân tích:** một dòng `order_id + order_item_id`.

**Câu hỏi:** Phí vận chuyển phân bố ra sao, liên hệ thế nào với sản phẩm/địa lý và khác nhau giữa các nhóm không?  
**Nguyên tắc:** chỉ đọc dữ liệu; không `dropna`, không xóa outlier, không xuất ghi đè; có số liệu rồi mới kết luận.

## 4.1. Data Source & Scope
**Ưu tiên** `../data/cleaned/olist_clean.csv` từ bước 2. Khi chưa có sẽ dùng `../data/raw/` để **khám phá tạm**, không coi là bản đã clean. Không phân tích payment/review (tránh bảng 1:N làm nhân dòng).

In [1]:
from pathlib import Path
import pandas as pd

# Dung du lieu da lam sach cua buoc 2 neu co. Khong sua file goc.
DATA_DIR = None
for path in [Path('../data'), Path('data'), Path('../../data')]:
    if (path / 'cleaned' / 'olist_clean.csv').exists() or (path / 'raw' / 'olist_order_items_dataset.csv').exists():
        DATA_DIR = path
        break

if DATA_DIR is None:
    raise FileNotFoundError('Khong thay data/cleaned/olist_clean.csv hoac data/raw/*.csv. Mo notebook trong notebooks/ cua repo.')

CLEAN_FILE = DATA_DIR / 'cleaned' / 'olist_clean.csv'
if CLEAN_FILE.exists():
    df = pd.read_csv(CLEAN_FILE)
    source = 'CLEANED (step 2)'
else:
    # Fallback de notebook van chay truoc khi nhom xuat du lieu clean.
    raw = DATA_DIR / 'raw'
    items = pd.read_csv(raw / 'olist_order_items_dataset.csv')
    orders = pd.read_csv(raw / 'olist_orders_dataset.csv',
                         usecols=['order_id','customer_id','order_status','order_purchase_timestamp'])
    customers = pd.read_csv(raw / 'olist_customers_dataset.csv',
                            usecols=['customer_id','customer_state'])
    products = pd.read_csv(raw / 'olist_products_dataset.csv',
                           usecols=['product_id','product_category_name','product_weight_g',
                                    'product_length_cm','product_height_cm','product_width_cm'])
    sellers = pd.read_csv(raw / 'olist_sellers_dataset.csv',
                          usecols=['seller_id','seller_state'])
    translation = pd.read_csv(raw / 'product_category_name_translation.csv')

    df = items.merge(orders, on='order_id', how='left', validate='m:1')
    df = df.merge(customers, on='customer_id', how='left', validate='m:1')
    df = df.merge(products, on='product_id', how='left', validate='m:1')
    df = df.merge(sellers, on='seller_id', how='left', validate='m:1')
    df = df.merge(translation, on='product_category_name', how='left', validate='m:1')
    source = 'RAW fallback (chua ap dung rule cleaning cua buoc 2)'

# Chuan hoa TEN COT chi trong RAM (neu buoc 2 dat ten ngan hon).
aliases = {
    'c_state': 'customer_state', 's_state': 'seller_state',
    'category': 'product_category_name',
    'category_english': 'product_category_name_english',
}
for old, new in aliases.items():
    if new not in df.columns and old in df.columns:
        df = df.rename(columns={old: new})

# Cot thoi gian: neu buoc 2 chua giu lai, lay them tu RAW (1 order -> 1 timestamp).
if 'order_purchase_timestamp' not in df.columns:
    orders_file = DATA_DIR / 'raw' / 'olist_orders_dataset.csv'
    if orders_file.exists():
        orders_time = pd.read_csv(orders_file,
                                  usecols=['order_id','order_purchase_timestamp'])
        df = df.merge(orders_time, on='order_id', how='left', validate='m:1')

required = ['order_id','order_item_id','freight_value','price','seller_id']
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f'Du lieu khong dung schema item-level, thieu cot: {missing}')

print('SOURCE:', source)
print('ROWS / COLS:', df.shape)
print('GRAIN: (order_id, order_item_id) | TARGET: freight_value')

SOURCE: CLEANED (step 2)
ROWS / COLS: (108937, 27)
GRAIN: (order_id, order_item_id) | TARGET: freight_value


## 4.2. Data Validation — đúng grain trước khi tính toán
**Mục đích:** Xác minh mỗi dòng vẫn là một item; nếu merge làm nhân dòng, các trung bình/tần suất sẽ sai. Kiểm tra cũng phân biệt `NaN`, `0`, âm và trùng khóa.

In [2]:
key_dup = df.duplicated(subset=['order_id', 'order_item_id']).sum()
freight = pd.to_numeric(df['freight_value'], errors='coerce')
checks = pd.DataFrame({
    'kiem_tra': ['So item', 'Trung khoa item', 'Thieu target', 'Target = 0', 'Target < 0'],
    'so_luong': [len(df), key_dup, freight.isna().sum(), freight.eq(0).sum(), freight.lt(0).sum()]
})
display(checks)
assert key_dup == 0, 'Loi grain: khoa item trung. Can quay lai buoc 2 kiem tra merge.'

,kiem_tra,so_luong
0,So item,108937
1,Trung khoa item,0
2,Thieu target,0
3,Target = 0,0
4,Target < 0,0


**Cách hiểu:** `freight_value = 0` chưa phải lỗi (có thể miễn phí vận chuyển); outlier cũng chưa phải lỗi. Notebook này **không tự xóa** các trường hợp đó.

## 4.3. Target Distribution — phí vận chuyển điển hình là bao nhiêu?
**Mục đích:** mô tả trung tâm, độ phân tán và tỷ lệ giá trị cực trị; ghi cả **n** và **%**, không chỉ dùng mean.

In [3]:
s = freight.dropna()
q1 = s.quantile(.25)
q3 = s.quantile(.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outlier = (s < lower) | (s > upper)

summary_target = pd.DataFrame({
    'n': [len(s)],
    'mean': [s.mean()], 'median': [s.median()],
    'std': [s.std()], 'Q1': [q1], 'Q3': [q3],
    'P95': [s.quantile(.95)], 'P99': [s.quantile(.99)],
    'max': [s.max()],
    'IQR_outliers': [outlier.sum()],
    'outliers_%': [100*outlier.mean()],
    'zero_%': [100*s.eq(0).mean()]
})
display(summary_target.round(2))
print(f'Mean = {s.mean():.2f}; median = {s.median():.2f}.')
print(f'IQR outliers = {outlier.sum():,} items ({100*outlier.mean():.2f}%).')

,n,mean,median,std,Q1,Q3,P95,P99,max,IQR_outliers,outliers_%,zero_%
0,108937,20.01,16.28,15.7,13.08,21.15,45.1,83.96,409.68,11349,10.42,0.0


Mean = 20.01; median = 16.28.
IQR outliers = 11,349 items (10.42%).


**Lập luận:** Nếu `mean > median`, dữ liệu có dấu hiệu bị kéo lên bởi phía giá trị lớn; cần **xem histogram** ở bước 5 để xác nhận độ lệch. IQR chỉ gắn cờ để kiểm tra, **không phải tiêu chí xóa tự động**.

## 4.4. Numerical Relationships — giá, cân nặng, thể tích
**Mục đích:** sàng lọc biến có thể liên quan đến `freight_value` trước mô hình. `price`, `weight`, `volume` đều có trước lúc giao; chưa suy diễn quan hệ nhân quả.

In [4]:
# Cac cot phuc vu EDA; khong phai preprocessing cho model.
if all(col in df.columns for col in ['product_length_cm','product_height_cm','product_width_cm']):
    df['volume_cm3'] = (df['product_length_cm'] *
                        df['product_height_cm'] * df['product_width_cm'])

if 'customer_state' in df.columns and 'seller_state' in df.columns:
    valid = df['customer_state'].notna() & df['seller_state'].notna()
    df['same_state'] = pd.NA
    df.loc[valid, 'same_state'] = (df.loc[valid, 'customer_state'] ==
                                   df.loc[valid, 'seller_state'])

if 'product_category_name_english' in df.columns:
    df['category_eda'] = df['product_category_name_english']
    if 'product_category_name' in df.columns:
        df['category_eda'] = df['category_eda'].fillna(df['product_category_name'])
elif 'product_category_name' in df.columns:
    df['category_eda'] = df['product_category_name']

if 'order_purchase_timestamp' in df.columns:
    df['purchase_date'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
    df['purchase_month'] = df['purchase_date'].dt.to_period('M')

print('EDA columns:', [c for c in ['volume_cm3','same_state','category_eda','purchase_month'] if c in df.columns])

EDA columns: ['volume_cm3', 'same_state', 'category_eda', 'purchase_month']


In [5]:
numeric_cols = [col for col in ['price', 'product_weight_g', 'volume_cm3', 'freight_value']
                if col in df.columns]
pearson = df[numeric_cols].corr(method='pearson')['freight_value'].round(3)
spearman = df[numeric_cols].corr(method='spearman')['freight_value'].round(3)
relations = pd.DataFrame({'pearson': pearson, 'spearman': spearman})
relations = relations.drop(index='freight_value')
display(relations)

for col in numeric_cols:
    if col != 'freight_value':
        print(f'{col}: n = {df[[col, "freight_value"]].dropna().shape[0]:,}')

,pearson,spearman
price,0.414,0.438
volume_cm3,0.584,0.369


price: n = 108,937
volume_cm3: n = 108,937


**Cách đọc:** Pearson đo quan hệ **tuyến tính**, Spearman đo quan hệ **đơn điệu theo thứ hạng**. Giá trị gần 0 **không chứng minh** không có bất kỳ quan hệ nào. Không dùng `freight_value` để tạo feature cho mô hình.

## 4.5. Categorical Analysis — khác nhau giữa các nhóm?
**Mục đích:** so sánh nhóm có đủ quan sát; nhóm có trung bình cao nhưng rất ít item có thể không ổn định.

In [6]:
# Top 5 seller theo SO ITEMS (khong phai so don)
top5 = df['seller_id'].value_counts().head(5).index
seller_stats = (df[df['seller_id'].isin(top5)]
                .groupby('seller_id')['freight_value']
                .agg(['size','mean','median'])
                .sort_values('size', ascending=False))
print('TOP 5 SELLERS BY ITEM COUNT')
display(seller_stats.round(2))

if 'same_state' in df.columns:
    print('SAME STATE VS DIFFERENT STATE')
    state_stats = (df.dropna(subset=['same_state'])
                   .groupby('same_state')['freight_value']
                   .agg(['size','mean','median']))
    display(state_stats.round(2))

if 'category_eda' in df.columns:
    category_stats = (df.groupby('category_eda')['freight_value']
                       .agg(['size','mean','median']))
    # Chi xep hang nhom co it nhat 100 items; nguong loc phuc vu SO SANH, khong xoa rows.
    category_stats = category_stats.loc[category_stats['size'] >= 100]
    print('TOP 8 CATEGORIES (n >= 100), BY MEDIAN FREIGHT')
    display(category_stats.sort_values('median', ascending=False).head(8).round(2))

TOP 5 SELLERS BY ITEM COUNT


,size,mean,median
seller_id,,,
6560211a19b47992c3666cc44a7e94c0,1991,13.70,14.10
4a3ca9315b744ce9f8e9374361493884,1947,17.61,15.70
1f50f920176fa81dab994f9023523100,1865,18.75,17.67
cc419e0650a3c5ba77189a1882b7556a,1704,14.45,14.34
da8622b14eb17ae2831f4ac5b9dab84a,1546,16.07,15.10


SAME STATE VS DIFFERENT STATE


,size,mean,median
same_state,,,
False,69330,23.74,18.26
True,39607,13.48,12.03


TOP 8 CATEGORIES (n >= 100), BY MEDIAN FREIGHT


,size,mean,median
category_eda,,,
computers,198,48.65,41.55
furniture_bedroom,103,42.01,36.18
home_appliances_2,230,44.48,35.90
office_furniture,1655,40.17,34.12
kitchen_dining_laundry_garden_furniture,273,42.20,31.10
furniture_living_room,443,37.43,29.04
agro_industry_and_commerce,206,27.37,23.50
industry_commerce_and_business,260,28.77,23.09


**Cách hiểu:** so sánh `median` cùng với `mean` và `size`. `same_state = True` chỉ nói cùng **bang**, không đo chính xác khoảng cách. Category không dịch được giữ tên gốc; missing vẫn là missing, không tự gán nhóm giả.

## 4.6. Time Analysis — có xu hướng theo tháng không?
**Mục đích:** xem mức freight và lượng items theo thời gian; không tính ngày giao **thực tế** làm đầu vào cho bài toán dự đoán trước khi giao.

In [7]:
if 'purchase_month' in df.columns:
    monthly = (df.dropna(subset=['purchase_month'])
               .groupby('purchase_month')['freight_value']
               .agg(['size','mean','median'])
               .reset_index())
    monthly['purchase_month'] = monthly['purchase_month'].astype(str)
    display(monthly.tail(12).round(2))
else:
    print('Khong co ngay mua; bo qua time analysis va kiem tra lai schema buoc 2.')

,purchase_month,size,mean,median
8,2017-09,4711,19.77,16.27
9,2017-10,5178,19.74,16.11
10,2017-11,8435,19.52,16.11
11,2017-12,6152,18.92,15.66
12,2018-01,7998,19.07,15.66
13,2018-02,7478,18.58,15.61
14,2018-03,7981,20.86,17.94
15,2018-04,7634,20.74,18.23
16,2018-05,7558,19.91,16.45
17,2018-06,6981,22.23,18.30


**Cách hiểu:** tháng ít item có mean kém ổn định; mức freight theo tháng còn phụ thuộc loại hàng, seller và vị trí. Tránh nói “phí ship tăng theo thời gian” chỉ dựa một bảng trung bình.

## 4.7. Findings & Handoff → Step 5
**Kết luận dựa trên bằng chứng (tự in từ dữ liệu):**

In [8]:
print(f'1. Target: {len(s):,} gia tri hop le; mean {s.mean():.2f}, median {s.median():.2f}.')
print(f'2. Zero freight: {s.eq(0).sum():,} items ({100*s.eq(0).mean():.2f}%). Khong mac dinh la loi.')
print(f'3. IQR flags: {outlier.sum():,} items ({100*outlier.mean():.2f}%). Can kiem tra, khong tu xoa.')
print('4. Quan he so: xem bang Pearson/Spearman va chart scatter/heatmap o buoc 5.')
print('5. Khac biet nhom: xem mean + median + size truoc khi giai thich.')
print('6. Step 5: dung bieu do de xac nhan cac nhan dinh tren; khong sua file cleaning.')

1. Target: 108,937 gia tri hop le; mean 20.01, median 16.28.
2. Zero freight: 0 items (0.00%). Khong mac dinh la loi.
3. IQR flags: 11,349 items (10.42%). Can kiem tra, khong tu xoa.
4. Quan he so: xem bang Pearson/Spearman va chart scatter/heatmap o buoc 5.
5. Khac biet nhom: xem mean + median + size truoc khi giai thich.
6. Step 5: dung bieu do de xac nhan cac nhan dinh tren; khong sua file cleaning.
